In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## I. Importing Libraries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader

## II. Device Selection

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:" , device)

Using device: cpu


## III. PIL image to Tensors

In [4]:
transform = transforms.ToTensor()

## IV. Training and Testing Data

In [5]:
train_data = MNIST(root="data" , download=True , transform=transform , train=True)
test_data = MNIST(root="data" , download=True , transform=transform , train=False)

100%|██████████| 9.91M/9.91M [00:56<00:00, 176kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 103kB/s]
100%|██████████| 1.65M/1.65M [00:05<00:00, 277kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.61MB/s]


## V. Loaders

In [6]:
train_loaders = DataLoader(train_data , shuffle=True , batch_size=64)
test_loaders = DataLoader(test_data , shuffle=False , batch_size=64)

## VI. Making Model Class

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1 , 16 , kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16 , 32 , kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(32 * 5 * 5 , 64),
            nn.ReLU(),
            nn.Linear(64 , 10)
        )
    def forward(self , x):
        x = self.features(x)
        x = x.view(x.size(0) , -1)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)

## VII. Defining Loss function and Optimizer

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters() , lr=0.001)

## VIII. Training

In [9]:
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images , labels in train_loaders:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs , labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")

Epoch 1/3 | Loss: 224.6419
Epoch 2/3 | Loss: 67.0807
Epoch 3/3 | Loss: 47.2111


## VIII. Testing

In [10]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images , labels in test_loaders:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        prediction = outputs.argmax(dim=1)



        correct += (prediction == labels).sum().item()
        total += labels.size(0)

## IX. Accuracy

In [11]:
accuracy = 100 * correct / total
print(f"Model Accuracy: {accuracy}%")

Model Accuracy: 98.31%


## X. Saving Model

In [12]:
torch.save(model.state_dict(), "simple CNN model.pth")

## XI. Testing Own Image

In [24]:
from PIL import Image
import torchvision.transforms.functional as F
img = Image.open("33.png").convert("L")
transform = transforms.Compose([
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5, ))
])
img = transform(img)
img = F.invert(img)
img = img.unsqueeze(0)
print(img.shape)
model.load_state_dict(torch.load("simple CNN model.pth"))
model.eval()
with torch.no_grad():
    output = model(img)
    pred = output.argmax(dim=1)
print(pred.item())

torch.Size([1, 1, 28, 28])
2
